In [ ]:

import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Authenticate and initialize GEE
ee.Authenticate()
ee.Initialize(project='ors-assignment-coimbotore')
print("✅ Google Earth Engine initialized successfully!")

✅ Google Earth Engine initialized successfully!


In [ ]:
states = ee.FeatureCollection("FAO/GAUL/2015/level1")
india_states = states.filter(ee.Filter.eq('ADM0_NAME', 'India'))

punjab = india_states.filter(ee.Filter.eq('ADM1_NAME', 'Punjab'))
haryana = india_states.filter(ee.Filter.eq('ADM1_NAME', 'Haryana'))

punjab_haryana = punjab.merge(haryana).union()

source_region = punjab_haryana.geometry()
districts = ee.FeatureCollection("FAO/GAUL/2015/level2")
india_districts = districts.filter(ee.Filter.eq('ADM0_NAME', 'India'))

ncr_districts = [

    'Gurgaon', 'Faridabad', 'Sonepat', 'Rohtak', 'Jhajjar',
    'Palwal', 'Nuh', 'Rewari', 'Panipat', 'Mahendragarh',
    'Bhiwani', 'Charkhi Dadri', 'Karnal', 'Jind',

    'Ghaziabad', 'Gautam Buddha Nagar', 'Meerut', 'Baghpat',
    'Hapur', 'Bulandshahr', 'Muzaffarnagar', 'Shamli',

    'Alwar', 'Bharatpur'
]

ncr_other = india_districts.filter(
    ee.Filter.inList('ADM2_NAME', ncr_districts)
)

delhi = india_districts.filter(
    ee.Filter.eq('ADM2_NAME', 'Delhi')
)

ncr = ncr_other.merge(delhi)

target_region = ncr.union().geometry()


In [ ]:
Map = geemap.Map()
Map.addLayer(source_region, {}, "Punjab-Haryana")
Map.addLayer(target_region, {}, "NCR")
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

# MODIS FIRE DATA

In [ ]:
def get_fire_data(start_date, end_date):

    fires = ee.ImageCollection("MODIS/061/MOD14A1") \
        .filterDate(start_date, end_date) \
        .map(lambda img: img.clip(source_region))


    def fire_stats(img):

        firemask = img.select('FireMask')

        fire_pixels = firemask.gte(7)

        fire_count = fire_pixels.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=source_region,
            scale=1000,
            maxPixels=1e13
        ).get('FireMask')


        frp = img.select('MaxFRP') \
            .updateMask(fire_pixels) \
            .reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=source_region,
                scale=1000,
                maxPixels=1e13
            ).get('MaxFRP')


        return ee.Feature(None,{
            'date': img.date().format(),
            'fire_count': fire_count,
            'total_frp': frp
        })


    fire_features = ee.FeatureCollection(
        fires.map(fire_stats)
    )

    fire_df = geemap.ee_to_df(fire_features)

    return fire_df
def get_burned_area(start_date, end_date):

    burned = ee.ImageCollection("MODIS/061/MCD64A1") \
        .filterDate(start_date, end_date) \
        .select('BurnDate') \
        .map(lambda img: img.clip(source_region))


    def burn_stats(img):

        burn_date = img.select('BurnDate')

        # burned pixels = BurnDate > 0
        burned_pixels = burn_date.gt(0)

        # convert burned pixels to area
        burned_area = burned_pixels.multiply(ee.Image.pixelArea())

        stats = burned_area.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=source_region,
            scale=500,
            maxPixels=1e13
        )

        return ee.Feature(None,{
            'date': img.date().format(),
            'burned_area': stats.get('BurnDate')
        })


    burn_features = ee.FeatureCollection(
        burned.map(burn_stats)
    )

    burn_df = geemap.ee_to_df(burn_features)

    # convert m² → km²
    burn_df['burned_area'] = burn_df['burned_area'] / 1e6

    return burn_df

# SENTINEL-5P POLLUTION

In [ ]:
def get_pollution_data(start_date, end_date):

    # CO
    co = ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_CO") \
        .filterDate(start_date, end_date) \
        .select('CO_column_number_density')

    # NO2
    no2 = ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_NO2") \
        .filterDate(start_date, end_date) \
        .select('tropospheric_NO2_column_number_density')

    # Aerosol Index
    ai = ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_AER_AI") \
        .filterDate(start_date, end_date) \
        .select('absorbing_aerosol_index')


    # Merge pollutant bands
    pollution = co.map(
        lambda img: img.addBands(
            no2.filterDate(img.date(), img.date().advance(1,'day')).mean()
        ).addBands(
            ai.filterDate(img.date(), img.date().advance(1,'day')).mean()
        )
    )


    def pollution_stats(img):

        stats = img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=target_region,
            scale=1000,
            maxPixels=1e13
        )

        return ee.Feature(None,{
            'date': img.date().format(),
            'co': stats.get('CO_column_number_density'),
            'no2': stats.get('tropospheric_NO2_column_number_density'),
            'aerosol_index': stats.get('absorbing_aerosol_index')
        })


    pollution_features = ee.FeatureCollection(
        pollution.map(pollution_stats)
    )

    pollution_df = geemap.ee_to_df(pollution_features)

    return pollution_df

# RETRIVE DATA

In [ ]:
years = list(range(2018, 2024))
all_X = []
all_Y = []
for year in years:

    print(f"Processing year: {year}")

    start_date = f"{year}-06-01"
    end_date = f"{year}-12-31"

    # Source-region data
    fire_df = get_fire_data(start_date, end_date)
    burn_df = get_burned_area(start_date, end_date)

    X_df = fire_df.merge(burn_df, on='date', how='outer')

    X_df['year'] = year

    # Target-region data
    Y_df = get_pollution_data(start_date, end_date)

    Y_df['year'] = year

    all_X.append(X_df)
    all_Y.append(Y_df)
X_df = pd.concat(all_X, ignore_index=True)
Y_df = pd.concat(all_Y, ignore_index=True)
X_df['date'] = pd.to_datetime(X_df['date'])
Y_df['date'] = pd.to_datetime(Y_df['date'])
X_df.info()
Y_df.info()

Processing year: 2018
Processing year: 2019
Processing year: 2020
Processing year: 2021
Processing year: 2022
Processing year: 2023
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1266 entries, 0 to 1265
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         1266 non-null   datetime64[ns]
 1   fire_count   1266 non-null   float64       
 2   total_frp    1266 non-null   float64       
 3   burned_area  42 non-null     float64       
 4   year         1266 non-null   int64         
dtypes: datetime64[ns](1), float64(3), int64(1)
memory usage: 49.6 KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17567 entries, 0 to 17566
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   aerosol_index  17494 non-null  float64       
 1   date           17567 non-null  datetime64[ns]
 2   no2            17133 non-null  f

# Compute Baseline

In [ ]:
fire_baseline = X_df[
    X_df['date'].dt.month.isin([6,7])
]

fire_baseline_vals = fire_baseline.groupby('year')[
    ['fire_count','total_frp','burned_area']
].mean()

pollution_baseline = Y_df[
    Y_df['date'].dt.month.isin([6,7])
]

pollution_baseline_vals = pollution_baseline.groupby('year')[
    ['co','no2','aerosol_index']
].mean()

print(fire_baseline_vals)
print(pollution_baseline_vals)

X_season = X_df[X_df['date'].dt.month.isin([9,10,11,12])]

Y_season = Y_df[Y_df['date'].dt.month.isin([9,10,11,12])]

X_season = X_season.merge(
    fire_baseline_vals,
    on='year',
    suffixes=('','_baseline')
)

Y_season = Y_season.merge(
    pollution_baseline_vals,
    on='year',
    suffixes=('','_baseline')
)

      fire_count   total_frp  burned_area
year                                     
2018    0.636901   87.266152     2.414958
2019    2.313533  646.275538     5.046026
2020    1.380392  264.712054    69.665963
2021    1.345034  480.250466     0.706836
2022    1.271038  173.069752    25.441706
2023    2.885246  575.508197    18.480715
            co       no2  aerosol_index
year                                   
2018  0.036743  0.000040      -0.609520
2019  0.036769  0.000045      -0.464943
2020  0.036893  0.000037      -1.108219
2021  0.036666  0.000048      -0.580493
2022  0.035888  0.000044      -0.007431
2023  0.035241  0.000039      -0.370364


In [ ]:
X_season['fire_anomaly'] = (
    X_season['fire_count'] - X_season['fire_count_baseline']
)

X_season['frp_anomaly'] = (
    X_season['total_frp'] - X_season['total_frp_baseline']
)

X_season['burn_anomaly'] = (
    X_season['burned_area'] - X_season['burned_area_baseline']
)

Y_season['co_anomaly'] = Y_season['co'] - Y_season['co_baseline']

Y_season['no2_anomaly'] = Y_season['no2'] - Y_season['no2_baseline']

Y_season['ai_anomaly'] = (
    Y_season['aerosol_index'] - Y_season['aerosol_index_baseline']
)
X_season['date'] = pd.to_datetime(X_season['date']).dt.date
Y_season['date'] = pd.to_datetime(Y_season['date']).dt.date

dataset = X_season.merge(Y_season, on='date', how='inner')
dataset.info()

dataset.to_csv("fire_pollution_dataset.csv", index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10102 entries, 0 to 10101
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   date                    10102 non-null  object 
 1   fire_count              10102 non-null  float64
 2   total_frp               10102 non-null  float64
 3   burned_area             342 non-null    float64
 4   year_x                  10102 non-null  int64  
 5   fire_count_baseline     10102 non-null  float64
 6   total_frp_baseline      10102 non-null  float64
 7   burned_area_baseline    10102 non-null  float64
 8   fire_anomaly            10102 non-null  float64
 9   frp_anomaly             10102 non-null  float64
 10  burn_anomaly            342 non-null    float64
 11  aerosol_index           10042 non-null  float64
 12  no2                     9934 non-null   float64
 13  co                      847 non-null    float64
 14  year_y                  10102 non-null

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

dataset.to_csv('/content/drive/MyDrive/fire_pollution_dataset.csv', index=False)

Mounted at /content/drive


# Clean Data

In [ ]:
dataset['date'] = pd.to_datetime(dataset['date'])
daily_dataset = dataset.groupby('date').agg({
    'fire_count': 'first',
    'total_frp': 'first',
    'burned_area': 'first',
    'fire_anomaly': 'first',
    'frp_anomaly': 'first',
    'burn_anomaly': 'first',
    'co': 'mean',
    'no2': 'mean',
    'aerosol_index': 'mean',
    'co_anomaly': 'mean',
    'no2_anomaly': 'mean',
    'ai_anomaly': 'mean'
}).reset_index()
daily_dataset['month'] = daily_dataset['date'].dt.to_period('M')

daily_dataset['burned_area'] = (
    daily_dataset.groupby('month')['burned_area']
    .transform(lambda x: x.ffill().bfill())
)

daily_dataset = daily_dataset.drop(columns=['month'])
daily_dataset['date'] = pd.to_datetime(daily_dataset['date'])

daily_dataset = daily_dataset.sort_values('date').reset_index(drop=True)
fire_cols = ['fire_count', 'total_frp' ]

daily_dataset[fire_cols] = daily_dataset[fire_cols].fillna(0)
daily_dataset['month'] = daily_dataset['date'].dt.to_period('M')

daily_dataset['burned_area'] = (
    daily_dataset.groupby('month')['burned_area']
    .transform(lambda x: x.ffill().bfill())
)

daily_dataset = daily_dataset.drop(columns=['month'])

fire_baseline_lookup = dataset[['date', 'fire_count_baseline']].drop_duplicates()
frp_baseline_lookup = dataset[['date', 'total_frp_baseline']].drop_duplicates()
burn_baseline_lookup = dataset[['date', 'burned_area_baseline']].drop_duplicates()

fire_baseline_lookup['date'] = pd.to_datetime(fire_baseline_lookup['date'])
frp_baseline_lookup['date'] = pd.to_datetime(frp_baseline_lookup['date'])
burn_baseline_lookup['date'] = pd.to_datetime(burn_baseline_lookup['date'])

fire_baseline = daily_dataset['date'].map(
    fire_baseline_lookup.set_index('date')['fire_count_baseline']
)

frp_baseline = daily_dataset['date'].map(
    frp_baseline_lookup.set_index('date')['total_frp_baseline']
)

burn_baseline = daily_dataset['date'].map(
    burn_baseline_lookup.set_index('date')['burned_area_baseline']
)

daily_dataset['fire_anomaly'] = daily_dataset['fire_count'] - fire_baseline

daily_dataset['frp_anomaly'] = daily_dataset['total_frp'] - frp_baseline

daily_dataset['burn_anomaly'] = daily_dataset['burned_area'] - burn_baseline
pollution_cols = ['co', 'no2', 'aerosol_index']

daily_dataset[pollution_cols] = (
    daily_dataset[pollution_cols]
    .interpolate(method='linear')
)
daily_dataset = daily_dataset.dropna()
daily_dataset.to_csv("fire_pollution_dataset_daily_cleaned.csv", index=False)

In [ ]:
dataset.to_csv('/content/drive/MyDrive/fire_pollution_dataset_daily_cleaned.csv', index=False)